# Colab Starter: Class Student Accent Training

This notebook runs the class-data pipeline fully in Colab: Drive mount, repo sync, metadata prep, speaker-disjoint splits, validation, feature extraction, and training.

In [ ]:
import re

import shlex

import subprocess

from pathlib import Path

import pandas as pd



def run(cmd, cwd=None, allow_fail=False):

    print(f'\n$ {cmd}')

    result = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)

    if result.stdout:

        print(result.stdout)

    if result.stderr:

        print(result.stderr)

    if result.returncode != 0 and not allow_fail:

        raise RuntimeError(f'Command failed ({result.returncode}): {cmd}')

    return result.returncode



def q(path_like):

    return shlex.quote(str(path_like))



REPO_URL = 'https://github.com/Saikumarlingaraju/Audio.git'

REPO_BRANCH = 'main'

PROJECT_DIR = Path('/content/iiitpro')



# Audio root in Drive must contain state folders directly (andhra_pradesh/, telangana/, ...).

DRIVE_AUDIO_ROOT = Path('/content/drive/MyDrive/iiitpro_audio/organized_by_state_clean')



# Safe defaults for first successful run.

RUN_HUBERT = False

RUN_ROBUST = False

BASELINE_EPOCHS = 20

ROBUST_EPOCHS = 30


In [ ]:
try:

    from google.colab import drive

    drive.mount('/content/drive')

except Exception as exc:

    print('Drive mount skipped (not in Colab?):', exc)



run('nvidia-smi || true', allow_fail=True)

run('apt-get update && apt-get install -y ffmpeg')

run('python -m pip install --upgrade pip setuptools wheel')



if PROJECT_DIR.exists():

    run('git -C {repo} fetch --all'.format(repo=q(PROJECT_DIR)))

    run('git -C {repo} checkout {branch}'.format(repo=q(PROJECT_DIR), branch=REPO_BRANCH))

    run('git -C {repo} pull origin {branch}'.format(repo=q(PROJECT_DIR), branch=REPO_BRANCH))

else:

    run('git clone --branch {branch} {url} {dest}'.format(branch=REPO_BRANCH, url=REPO_URL, dest=q(PROJECT_DIR)))



try:

    run('pip install -r requirements.txt', cwd=str(PROJECT_DIR))

except RuntimeError as exc:

    print('requirements.txt install failed in Colab. Falling back to a Colab-safe dependency set (without fairseq).')

    print('Original error:', exc)

    run('pip install --upgrade --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio')

    run(

        'pip install '

        '"transformers>=4.30.0" "librosa>=0.10.0" "soundfile>=0.12.0" '

        '"pydub>=0.25.1" "datasets>=2.14.0" "huggingface-hub>=0.16.0" '

        '"numpy>=1.24.0" "scipy>=1.10.0" "scikit-learn>=1.3.0" "pandas>=2.0.0" '

        '"matplotlib>=3.7.0" "seaborn>=0.12.0" "tqdm>=4.65.0" '

        '"pyyaml>=6.0" "python-dotenv>=1.0.0" "joblib>=1.3.0" '

        '"flask>=2.3.0" "flask-cors>=4.0.0" "gunicorn>=21.0.0" '

        '"psycopg2-binary>=2.9.9"'

    )


In [ ]:
CLASS_DATA_DIR = PROJECT_DIR / 'data/collections/class_students/organized_by_state_clean'

METADATA_ALL = CLASS_DATA_DIR / 'metadata_all.csv'

METADATA_READY = CLASS_DATA_DIR / 'metadata_training_ready.csv'

SPLIT_DIR = PROJECT_DIR / 'data/splits/class_students'

MFCC_FEATURE_DIR = PROJECT_DIR / 'data/features/class_students/mfcc'

HUBERT_FEATURE_DIR = PROJECT_DIR / 'data/features/class_students/hubert'



if not DRIVE_AUDIO_ROOT.exists():

    raise FileNotFoundError(f'Drive audio folder missing: {DRIVE_AUDIO_ROOT}')



CLASS_DATA_DIR.mkdir(parents=True, exist_ok=True)



if not METADATA_ALL.exists():

    print('metadata_all.csv not found in repo. Reconstructing from Drive audio folders...')

    ext_to_mime = {

        '.webm': 'audio/webm',

        '.wav': 'audio/wav',

        '.mp3': 'audio/mpeg',

        '.ogg': 'audio/ogg',

        '.m4a': 'audio/mp4',

        '.flac': 'audio/flac',

    }



    rows = []

    for state_dir in sorted([p for p in DRIVE_AUDIO_ROOT.iterdir() if p.is_dir()]):

        state = state_dir.name.strip().lower()

        for student_dir in sorted([p for p in state_dir.iterdir() if p.is_dir()]):

            parts = student_dir.name.split('__', 1)

            student_id = parts[0].strip().lower() if parts else 'unknown'

            student_name = parts[1].replace('_', ' ').strip() if len(parts) > 1 else student_id



            for submission_dir in sorted([p for p in student_dir.iterdir() if p.is_dir()]):

                submission_id = submission_dir.name.strip()

                for audio_file in sorted([p for p in submission_dir.iterdir() if p.is_file()]):

                    slot_match = re.search(r'prompt_(\d+)', audio_file.stem.lower())

                    slot = int(slot_match.group(1)) if slot_match else None

                    size = audio_file.stat().st_size

                    rel_path = audio_file.relative_to(DRIVE_AUDIO_ROOT).as_posix()

                    rows.append({

                        'filepath': rel_path,

                        'filename': audio_file.name,

                        'speaker_id': student_id,

                        'student_name': student_name,

                        'student_id': student_id,

                        'native_language': state,

                        'native_language_other': '',

                        'hometown': '',

                        'state_group': state,

                        'submission_id': submission_id,

                        'slot': slot,

                        'text': '',

                        'mime_type': ext_to_mime.get(audio_file.suffix.lower(), 'audio/webm'),

                        'size_bytes': size,

                        'is_tiny': 1 if size <= 10 else 0,

                        'age_group': 'adult',

                        'utterance_type': 'read',

                        'split': 'unassigned',

                        'source': 'drive_reconstructed',

                    })



    if not rows:

        raise RuntimeError(f'No audio files found under {DRIVE_AUDIO_ROOT}')



    pd.DataFrame(rows).to_csv(METADATA_ALL, index=False)

    print(f'Reconstructed metadata: {METADATA_ALL} (rows={len(rows)})')



print('Metadata source:', METADATA_ALL)

print('Drive audio root:', DRIVE_AUDIO_ROOT)


In [ ]:
run(
    'python scripts/prepare_class_metadata_for_training.py ' +
    '--metadata {metadata} --audio_root {audio_root} --output {output} --min_bytes 1000 --strict'.format(
        metadata=q(METADATA_ALL),
        audio_root=q(DRIVE_AUDIO_ROOT),
        output=q(METADATA_READY),
    ),
    cwd=str(PROJECT_DIR),
)

run(
    'python src/data/create_splits.py ' +
    '--metadata {metadata} --output_dir {output_dir} --train_ratio 0.7 --dev_ratio 0.15 --test_ratio 0.15'.format(
        metadata=q(METADATA_READY),
        output_dir=q(SPLIT_DIR),
    ),
    cwd=str(PROJECT_DIR),
)

run(
    'python scripts/validate_class_dataset.py ' +
    '--metadata {metadata} --audio_root {audio_root} --split_dir {split_dir}'.format(
        metadata=q(METADATA_READY),
        audio_root=q(DRIVE_AUDIO_ROOT),
        split_dir=q(SPLIT_DIR),
    ),
    cwd=str(PROJECT_DIR),
)

In [ ]:
run(
    'python src/features/mfcc_extractor.py ' +
    '--metadata {metadata} --audio_dir {audio_dir} --output_dir {output_dir}'.format(
        metadata=q(METADATA_READY),
        audio_dir=q(DRIVE_AUDIO_ROOT),
        output_dir=q(MFCC_FEATURE_DIR),
    ),
    cwd=str(PROJECT_DIR),
)

run(
    'python train_simple.py ' +
    '--features_path {features} --train_csv {train_csv} --val_csv {val_csv} --test_csv {test_csv} ' +
    '--output_dir {output_dir} --num_epochs {epochs} --batch_size 8'.format(
        features=q(MFCC_FEATURE_DIR / 'mfcc_stats.pkl'),
        train_csv=q(SPLIT_DIR / 'train.csv'),
        val_csv=q(SPLIT_DIR / 'dev.csv'),
        test_csv=q(SPLIT_DIR / 'test.csv'),
        output_dir=q(PROJECT_DIR / 'experiments/class_students_colab_mfcc'),
        epochs=BASELINE_EPOCHS,
    ),
    cwd=str(PROJECT_DIR),
)

In [ ]:
if RUN_HUBERT:
    run(
        'python src/features/hubert_extractor.py ' +
        '--metadata {metadata} --audio_dir {audio_dir} --output_dir {output_dir} --extract_layer 3 --pooling mean'.format(
            metadata=q(METADATA_READY),
            audio_dir=q(DRIVE_AUDIO_ROOT),
            output_dir=q(HUBERT_FEATURE_DIR),
        ),
        cwd=str(PROJECT_DIR),
    )

if RUN_ROBUST:
    run(
        'python train_robust.py ' +
        '--features_path {features} --train_csv {train_csv} --val_csv {val_csv} --test_csv {test_csv} ' +
        '--output_dir {output_dir} --model_type robust_mlp --num_epochs {epochs} --batch_size 8'.format(
            features=q(HUBERT_FEATURE_DIR / 'hubert_layer3_mean.pkl'),
            train_csv=q(SPLIT_DIR / 'train.csv'),
            val_csv=q(SPLIT_DIR / 'dev.csv'),
            test_csv=q(SPLIT_DIR / 'test.csv'),
            output_dir=q(PROJECT_DIR / 'experiments/class_students_colab_hubert_robust'),
            epochs=ROBUST_EPOCHS,
        ),
        cwd=str(PROJECT_DIR),
    )

In [ ]:
RUN_EXPORT_DIR = Path('/content/drive/MyDrive/iiitpro_runs/class_students')
RUN_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

run('cp -r {src} {dst} || true'.format(src=q(PROJECT_DIR / 'experiments/class_students_colab_mfcc'), dst=q(RUN_EXPORT_DIR)))
run('cp -r {src} {dst} || true'.format(src=q(PROJECT_DIR / 'experiments/class_students_colab_hubert_robust'), dst=q(RUN_EXPORT_DIR)))
run('cp -r {src} {dst} || true'.format(src=q(SPLIT_DIR), dst=q(RUN_EXPORT_DIR)))

print('Artifacts copied to:', RUN_EXPORT_DIR)